# Descriptive Statistics: Sleep Features

In [ ]:
%load_ext autoreload
%autoreload 2

import sys

from pathlib import Path
notebook_dir = Path().resolve()
sys.path.append(str(notebook_dir.parent))        # ../
sys.path.append(str(notebook_dir.parent.parent)) # ../../


import pandas as pd
import numpy as np
from scipy import stats
from datetime import datetime
import matplotlib.pyplot as plt



## Functions

In [ ]:
def check_normality(data, feature_name="Feature", ax=None):
    """
    Generate Q-Q plot and run Shapiro-Wilk test for a given array/series.
    
    Parameters
    ----------
    data : array-like
        Numeric data to test (NaNs are dropped automatically).
    feature_name : str
        Label used in the plot title and printed output.
    ax : matplotlib axis, optional
        If provided, plots on this axis (useful for subplots/loops).
        Otherwise creates its own figure.
    
    Returns
    -------
    dict with W statistic, p-value, and a boolean for normality at alpha=0.05
    """
    data = pd.Series(data).dropna().values

    # Shapiro-Wilk test
    W, p_value = stats.shapiro(data)
    is_normal = p_value > 0.05

    # Q-Q plot
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 5))
    stats.probplot(data, dist="norm", plot=ax)
    ax.set_title(f"Q-Q Plot: {feature_name}\nShapiro-Wilk W={W:.3f}, p={p_value:.4f}")

    print(f"{feature_name}: W={W:.4f}, p={p_value:.4f} -> "
          f"{'Normal (fail to reject H0)' if is_normal else 'Non-normal (reject H0)'}")

    return {"feature": feature_name, "W": W, "p_value": p_value, "is_normal": is_normal}



# --- Example: loop over multiple features in a feature matrix ---
def check_normality_batch(df, feature_cols, ncols=4):
    n = len(feature_cols)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
    axes = np.array(axes).reshape(-1)

    results = []
    for i, col in enumerate(feature_cols):
        data = df[col].dropna()
        W, p_value = stats.shapiro(data)
        skewness = stats.skew(data, bias=False)

        if abs(skewness) < 0.5:
            direction = "symmetric"
        elif skewness >= 0.5:
            direction = "right-skewed"
        else:
            direction = "left-skewed"

        stats.probplot(data, dist="norm", plot=axes[i])
        axes[i].set_title(f"{col}\nW={W:.3f}, p={p_value:.4f}\nskew={skewness:.2f} ({direction})")

        results.append({
            "feature": col, "W": W, "p_value": p_value,
            "is_normal": p_value > 0.05,
            "skewness": skewness, "direction": direction
        })

    for j in range(len(feature_cols), len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()
    return pd.DataFrame(results)

## Get Data

In [ ]:
df_sleep = pd.read_csv('../../output/1_feature_extraction/df_features_sleep_2026-07-08.csv')

# Convert time columns to seconds
time_columns = ['median_sleep_onset', 'median_midpoint', 'median_wakeup']
for col in time_columns:
    df_sleep[col + '_seconds'] = pd.to_datetime(df_sleep[col], format='mixed').dt.time.apply(
        lambda x: x.hour * 3600 + x.minute * 60 + x.second
    )
display(df_sleep[time_columns + [col + '_seconds' for col in time_columns]].head())

# Midnight crossover adjustment
df_sleep['median_sleep_onset_seconds_adjusted'] = df_sleep['median_sleep_onset_seconds'].apply(
    lambda x: x + 24 * 3600 if x < 12 * 3600 else x
)
df_sleep['median_midpoint_seconds_adjusted'] = df_sleep['median_midpoint_seconds'].apply(
    lambda x: x + 24 * 3600 if x < 12 * 3600 else x
)


# Keep as datetime for plotting (plotly handles datetime axes natively)
df_sleep['median_onset_plot'] = df_sleep['median_sleep_onset_seconds_adjusted'].apply(
    lambda x: pd.Timestamp('2000-01-01') + pd.Timedelta(seconds=x)
)

df_sleep['median_midpoint_plot'] = df_sleep['median_midpoint_seconds_adjusted'].apply(
    lambda x: pd.Timestamp('2000-01-01') + pd.Timedelta(seconds=x)
)
df_sleep['median_wakeup_plot'] = df_sleep['median_wakeup_seconds'].apply(
    lambda x: pd.Timestamp('2000-01-01') + pd.Timedelta(seconds=x)
)

def seconds_to_clock_hhmm(seconds):
    if pd.isna(seconds):
        return ""
    seconds = int(seconds) % (24 * 3600)
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    return f"{hours:02d}:{minutes:02d}"

# Calculate descriptive statistics

In [ ]:

df_stats = df_sleep.copy()
display(df_stats.columns)
df_stats = df_stats[['study_id', 'median_sleep_duration', 'median_sleep_onset', 'median_wakeup', 'median_midpoint', 'median_waso', 'median_awake', 'median_ser']]
display(df_stats.columns)
display(df_stats.head())


#convert string to datetime
df_stats['median_sleep_onset'] = pd.to_datetime(df_stats['median_sleep_onset'], format='mixed').dt.time
df_stats['median_wakeup'] = pd.to_datetime(df_stats['median_wakeup'], format='mixed').dt.time
df_stats['median_midpoint'] = pd.to_datetime(df_stats['median_midpoint'], format='mixed').dt.time
# Convert sleep_onset time to seconds since midnight for calculation
df_stats['onset_seconds'] = df_stats['median_sleep_onset'].apply(
    lambda x: x.hour * 3600 + x.minute * 60 + x.second
)

# Handle midnight crossover (adjust times before midday to be "past midnight")
df_stats['onset_seconds_adjusted'] = df_stats['onset_seconds'].apply(
    lambda x: x + 24 * 3600 if x < 12 * 3600 else x
)
# Convert midpoint_time to seconds
df_stats['midpoint_seconds'] = df_stats['median_midpoint'].apply(
    lambda x: x.hour * 3600 + x.minute * 60 + x.second
)

# Midnight crossover - threshold at 12:00 (midpoints before noon are past midnight)
df_stats['midpoint_seconds_adjusted'] = df_stats['midpoint_seconds'].apply(
    lambda x: x + 24 * 3600 if x < 12 * 3600 else x
)
df_stats['wakeup_seconds'] = df_stats['median_wakeup'].apply(
    lambda x: x.hour * 3600 + x.minute * 60 + x.second
)
#drop onset_seconds column
df_stats = df_stats.drop(columns=['onset_seconds', 'midpoint_seconds'])
# calculate mean, median, standard deviation, min, max
df_stats = df_stats.describe().transpose().reset_index()

def seconds_to_clock_time(seconds):
    """Wrap seconds back into a 24h clock time (for actual time-of-day values)."""
    return (pd.Timestamp('00:00:00') + pd.Timedelta(seconds=seconds % (24 * 3600))).time()

def seconds_to_duration_str(seconds):
    return seconds/60

time_cols = ['mean', 'min', '25%', '50%', '75%', 'max']
# cast these columns to object dtype so they can hold time/str values
df_stats[time_cols] = df_stats[time_cols].astype(object)
for index, row in df_stats.iterrows():
    if row['index'] == 'onset_seconds_adjusted' or row['index'] == 'midpoint_seconds_adjusted' or row['index'] == 'wakeup_seconds':
        for col in time_cols:
            df_stats.at[index, col] = seconds_to_clock_time(row[col])
        # std is a spread, not a point in time — format as duration instead
        df_stats.at[index, 'std'] = seconds_to_duration_str(row['std'])




display(df_stats)

date = datetime.now().strftime("%Y-%m-%d")
df_stats.to_csv(f'../../output/2_descriptive_stats/df_features_sleep_stats_{date}.csv', index=False)

### Q-Q Plot & Shapiro-Wilk Test   

In [ ]:
sleep_feature_cols = ['median_sleep_duration', 'median_waso', 'median_awake', 'median_sleep_onset_seconds_adjusted', 'median_midpoint_seconds_adjusted', 'median_wakeup_seconds', 'median_ser']
sleep_summary = check_normality_batch(df_sleep, sleep_feature_cols, ncols=6)
print(sleep_summary, f"\n")
date = datetime.now().strftime("%Y-%m-%d")
sleep_summary.to_csv(f"../../output/2_descriptive_stats/sleep_normality_summary_{date}.csv", index=False)